# Player Categorizing Model 
### K-Means Clustering via MapReduce (Hadoop Streaming)
**Labels:** Elite | Star | Reliable Player | Squad Player | Low Impact

**Pipeline Overview:**
1. Environment Setup & Data Loading
2. Feature Engineering
3. Preprocessing for MapReduce
4. MapReduce K-Means (Hadoop Streaming / Local Simulation)
5. Label Assignment
6. Evaluation & Visualization

In [ ]:
%pip install pandas scikit-learn matplotlib seaborn

## Step 1: Environment Setup and Data Loading

In [ ]:
import os
import re
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


DATA_DIR = 'Data'
OUTPUT_DIR = 'PreProcessedData'

if os.path.exists(OUTPUT_DIR):
    import shutil
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

END_SEASON = 2026

dataset = {
    'player_profiles': f'{DATA_DIR}/player_profiles/player_profiles.csv',
    'transfer_history': f'{DATA_DIR}/transfer_history/transfer_history.csv',
    'player_performances': f'{DATA_DIR}/player_performances/player_performances.csv',
    'player_national_performances': f'{DATA_DIR}/player_national_performances/player_national_performances.csv',
}

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
Columns = {
        'player_profiles': ['player_id', 'player_slug'],
        'transfer_history': ['player_id', 'transfer_date', 'from_team_id', 'to_team_id'],
        'player_performances': ['player_id', 'season_name', 'nb_in_group', 'nb_on_pitch', 'goals', 
                               'assists', 'own_goals', 'yellow_cards', 'second_yellow_cards', 
                               'direct_red_cards', 'penalty_goals', 'minutes_played', 
                               'goals_conceded', 'clean_sheets'],
        'player_national_performances': ['player_id', 'matches', 'goals'],
        'player_injuries': ['player_id', 'season_name', 'days_missed', 'games_missed']
    }

def is_date_column(col_name):
    date_keywords = ['date', 'day', 'dob', 'from', 'to', 'since', 'until', 'start', 'end']
    lower = col_name.lower()
    return any(keyword in lower for keyword in date_keywords)

def clean_string_series(series):
    return (
        series.astype(str)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

def coerce_numeric(series):
    cleaned = (
        series.astype(str)
        .str.replace(',', '', regex=False)
        .str.replace(r'[^0-9.+-]', '', regex=True)
    )
    return pd.to_numeric(cleaned, errors='coerce')

def summarize_zero_share(df, column_name, label):
    if column_name not in df.columns:
        print(f'⚠ {label}: column {column_name} not found')
        return {'zero_count': None, 'valid_count': None, 'percentage': None}

    numeric_series = pd.to_numeric(df[column_name], errors='coerce')
    valid_mask = numeric_series.notna()
    valid_count = int(valid_mask.sum())
    zero_count = int((numeric_series[valid_mask] == 0).sum())
    zero_percentage = (zero_count / valid_count * 100) if valid_count else 0.0

    print(f'• {label}: {zero_count:,}/{valid_count:,} rows are zero ({zero_percentage:.2f}%)')
    return {
        'zero_count': zero_count,
        'valid_count': valid_count,
        'percentage': zero_percentage,
    }

def extract_season_id_from_transfer_date(df):
    df = df.copy()
    df['transfer_date'] = pd.to_datetime(df['transfer_date'], errors='coerce')
    df = df.dropna(subset=['transfer_date'])
    df['season_id'] = df['transfer_date'].dt.year
    df.loc[df['transfer_date'].dt.month != 1, 'season_id'] += 1
    df['season_id'] = df['season_id'].astype(int)
    df = df[df['season_id'] <= END_SEASON] 
    return df

def extract_season_year_from_season_name(df):
    df = df.copy()
    if 'season_name' not in df.columns:
        return df
    season_raw = df['season_name'].astype('string').str.strip().dropna()
    season_pair = season_raw.str.extract(r'(?P<start>\d{2})[/-](?P<end>\d{2})')
    df['season_year'] = pd.to_numeric(season_pair['end'], errors='coerce') + 2000
    df = df.dropna(subset=['season_year'])
    df['season_year'] = df['season_year'].astype(int)
    df = df.drop(columns=['season_name'], errors='ignore')
    df = df[df['season_year'] <= END_SEASON]
    return df


def preprocess_table(name, path):
    df = pd.read_csv(path)
    df = df[Columns.get(name, [])]
    
    for col_name in df.columns:
        if df[col_name].dtype == 'object':
            if is_date_column(col_name):
                df[col_name] = pd.to_datetime(df[col_name], errors='coerce', infer_datetime_format=True)
            elif col_name == 'season_name':
                df[col_name] = clean_string_series(df[col_name])
            else:
                numeric_candidate = coerce_numeric(df[col_name])
                numeric_ratio = numeric_candidate.notna().mean()
                if numeric_ratio >= 0.7:
                    df[col_name] = numeric_candidate
                else:
                    df[col_name] = clean_string_series(df[col_name])

    if name == 'transfer_history':
        df = extract_season_id_from_transfer_date(df)
    elif name == 'player_performances':
        df = extract_season_year_from_season_name(df)

    output_path = os.path.join(OUTPUT_DIR, name, f'{name}.csv')
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f'✓ {name}: {len(df)} rows saved')

    return df

dfs = {}
for name, path in dataset.items():
    if os.path.exists(path):
        dfs[name] = preprocess_table(name, path)
    else:
        print(f'⚠ Warning: {path} not found')

print('\n✓ All needed data loaded and preprocessed')

## Step 2: Feature Engineering
Create aggregated player features from historical performance and transfer data.

In [ ]:
def build_last_n_transfer_features(transfer_df, n=3):
    if transfer_df.empty:
        return pd.DataFrame(columns=['player_id'])

    working = transfer_df.copy()
    working['transfer_date'] = pd.to_datetime(working['transfer_date'], errors='coerce')
    if 'transfer_fee' in working.columns:
        working['transfer_fee'] = pd.to_numeric(working['transfer_fee'], errors='coerce')
    if 'value_at_transfer' in working.columns:
        working['value_at_transfer'] = pd.to_numeric(working['value_at_transfer'], errors='coerce')

    working = working.sort_values(['player_id', 'transfer_date'], ascending=[True, False], na_position='last')
    feature_columns = ['from_team_id', 'to_team_id']
    available_columns = [column for column in feature_columns if column in working.columns]

    feature_frames = []
    for move_rank in range(1, n + 1):
        ranked = working.groupby('player_id').nth(move_rank - 1).reset_index()
        if ranked.empty:
            continue
        ranked = ranked[['player_id'] + available_columns].copy()
        ranked = ranked.rename(columns={column: f'move_{move_rank}_{column}' for column in available_columns})
        feature_frames.append(ranked)

    if not feature_frames:
        return pd.DataFrame(columns=['player_id'])

    merged = feature_frames[0]
    for frame in feature_frames[1:]:
        merged = merged.merge(frame, on='player_id', how='outer')

    return merged

def build_total_performance_features(player_perf_df):
    exclude_columns = {
        'player_id', 'season_name', 'season_year', 'competition_id',
        'competition_name', 'team_id', 'team_name',
    }
    numeric_columns = [
        column for column in player_perf_df.columns
        if column not in exclude_columns and pd.api.types.is_numeric_dtype(player_perf_df[column])
    ]

    if not numeric_columns:
        return pd.DataFrame(columns=['player_id'])

    totals = player_perf_df.groupby('player_id')[numeric_columns].sum(min_count=1).reset_index()
    totals = totals.rename(columns={column: f'{column}_total' for column in numeric_columns})
    return totals

    
# 1. Base Profiles
profiles_df = dfs['player_profiles'].copy()

# 2. Extract the last 3 moves for each player
print("Extracting last 3 transfers...")
transfer_df = dfs['transfer_history'].copy()
last_3_moves_df = build_last_n_transfer_features(transfer_df, n=3)

# 3. Calculate total club performance (sum over all seasons)
print("Calculating total performances...")
perf_df = dfs['player_performances'].copy()
total_perf_df = build_total_performance_features(perf_df)

# 4. Calculate total national performance
print("Calculating national performances...")
nat_df = dfs['player_national_performances'].copy()
nat_perf_df = nat_df.groupby('player_id')[['matches', 'goals']].sum().reset_index()
nat_perf_df = nat_perf_df.rename(columns={
    'matches': 'national_matches_total', 
    'goals': 'national_goals_total'
})

# 5. Merge all features together
print("Merging features...")
X_df = profiles_df.copy()
X_df = X_df.merge(last_3_moves_df, on='player_id', how='left')
X_df = X_df.merge(total_perf_df, on='player_id', how='left')
X_df = X_df.merge(nat_perf_df, on='player_id', how='left')

# 6. Fill missing performance values with 0
perf_cols = [c for c in total_perf_df.columns if c != 'player_id']
nat_cols = [c for c in nat_perf_df.columns if c != 'player_id']
cols_to_fill = perf_cols + nat_cols
X_df[cols_to_fill] = X_df[cols_to_fill].fillna(0)

print(f"\n✓ Feature matrix shape: {X_df.shape}")
X_df.to_csv(f'{OUTPUT_DIR}/final_model_features.csv', index=False)
print(f"✓ Features saved to {OUTPUT_DIR}/final_model_features.csv")
X_df.head()

## Step 3: Preprocessing for MapReduce K-Means

Select the most meaningful numeric features for clustering, apply **StandardScaler**, then export to **TSV format** — the input format expected by `mapper.py`.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── Select clustering features ────────────────────────────────────────────────
# These capture the four performance dimensions that define a player's tier:
#   • Volume   → minutes on pitch, appearances
#   • Attack   → goals, assists, penalty goals
#   • Defense  → clean sheets, goals conceded
#   • Discipline → cards
#   • National → international caps & goals

CANDIDATE_FEATURES = [
    'nb_on_pitch_total',      # total appearances
    'minutes_played_total',   # minutes on pitch
    'goals_total',            # club goals
    'assists_total',          # club assists
    'penalty_goals_total',    # penalty contributions
    'clean_sheets_total',     # defensive reliability
    'goals_conceded_total',   # defensive pressure (inverted signal)
    'yellow_cards_total',     # discipline
    'national_matches_total', # international exposure
    'national_goals_total',   # international impact
]

# Keep only columns that actually exist in X_df
CLUSTERING_FEATURES = [f for f in CANDIDATE_FEATURES if f in X_df.columns]
print(f"✓ Using {len(CLUSTERING_FEATURES)} clustering features:")
for f in CLUSTERING_FEATURES:
    print(f"   • {f}")

# ── Build feature matrix ──────────────────────────────────────────────────────
feature_matrix = X_df[['player_id'] + CLUSTERING_FEATURES].copy()
feature_matrix[CLUSTERING_FEATURES] = feature_matrix[CLUSTERING_FEATURES].fillna(0)

# Remove players with zero activity (all clustering features are 0)
active_mask = feature_matrix[CLUSTERING_FEATURES].sum(axis=1) > 0
feature_matrix = feature_matrix[active_mask].reset_index(drop=True)
print(f"\n✓ Active players (non-zero stats): {len(feature_matrix):,}")

# ── Scale features ────────────────────────────────────────────────────────────
scaler = StandardScaler()
scaled_values = scaler.fit_transform(feature_matrix[CLUSTERING_FEATURES])
scaled_df = pd.DataFrame(scaled_values, columns=CLUSTERING_FEATURES)
scaled_df.insert(0, 'player_id', feature_matrix['player_id'].values)

print("\n✓ Feature scaling applied (StandardScaler)")
scaled_df.describe().round(3)

In [ ]:
import os

# ── Directories ───────────────────────────────────────────────────────────────
MR_DIR   = 'MapReduceKMeans'          # root working directory for MapReduce
INPUT_DIR  = f'{MR_DIR}/input'        # HDFS-style input (TSV data points)
ITER_DIR   = f'{MR_DIR}/iterations'  # centroid snapshots per iteration

for d in [INPUT_DIR, ITER_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Export data points as TSV (tab-separated, no header, no player_id) ────────
# mapper.py reads: feature1 \t feature2 \t ... \t featureN  (one point per line)
TSV_PATH = f'{INPUT_DIR}/data.tsv'
scaled_df[CLUSTERING_FEATURES].to_csv(TSV_PATH, sep='\t', index=False, header=False)

# Save player_id mapping so we can re-attach labels after clustering
scaled_df[['player_id']].to_csv(f'{MR_DIR}/player_id_index.csv', index=True)

print(f"✓ TSV data written  → {TSV_PATH}  ({len(scaled_df):,} rows × {len(CLUSTERING_FEATURES)} features)")
print(f"✓ Player ID index   → {MR_DIR}/player_id_index.csv")

# Quick sanity check — peek at first 3 lines
print("\nFirst 3 lines of data.tsv:")
with open(TSV_PATH) as f:
    for i, line in enumerate(f):
        if i == 3: break
        print(' | '.join(line.strip().split('\t')[:5]), '...')

## Step 4: MapReduce K-Means (K = 5)

We run **iterative MapReduce** using the exact `mapper.py` and `reducer.py` scripts via **Hadoop Streaming** or a **local subprocess simulation** (same code path — just swap the runner).

### How it works
```
Iteration 1
  centroids.txt  ──►  mapper.py  ──►  (cluster_id \t features)  ──►  reducer.py  ──►  new centroids.txt
  
Repeat until centroids converge (or max_iter reached)
```

### Initialisation
We use **K-Means++ seeding** to pick 5 well-spread initial centroids — this drastically reduces the number of iterations needed.

In [ ]:
import numpy as np
import random

K            = 5          # elite, star, reliable, squad, low-impact
MAX_ITER     = 20         # safety cap
CONVERGENCE  = 1e-4       # stop when centroid shift < this threshold
RANDOM_SEED  = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

data_points = scaled_df[CLUSTERING_FEATURES].values   # shape (N, D)

# ── K-Means++ initialisation ──────────────────────────────────────────────────
def kmeans_plus_plus_init(X, k, seed=42):
    """Choose k initial centroids using K-Means++ seeding."""
    rng = np.random.default_rng(seed)
    n   = X.shape[0]

    # 1st centroid: random point
    first_idx  = rng.integers(0, n)
    centroids  = [X[first_idx]]

    for _ in range(1, k):
        # Squared distance from each point to its nearest centroid
        dists = np.array([
            min(np.sum((x - c) ** 2) for c in centroids)
            for x in X
        ])
        probs    = dists / dists.sum()                  # proportional probability
        next_idx = rng.choice(n, p=probs)
        centroids.append(X[next_idx])

    return np.array(centroids)   # shape (k, D)


initial_centroids = kmeans_plus_plus_init(data_points, K, seed=RANDOM_SEED)
print(f"✓ K-Means++ initialised {K} centroids  (shape: {initial_centroids.shape})")


# ── centroids.txt writer ──────────────────────────────────────────────────────
def write_centroids(centroids: np.ndarray, path: str):
    """Write centroids in the format expected by mapper.py:
       cluster_id \t f1,f2,...,fN
    """
    with open(path, 'w') as f:
        for idx, centroid in enumerate(centroids):
            coords = ','.join(map(str, centroid))
            f.write(f"{idx}\t{coords}\n")


# ── centroids.txt reader ──────────────────────────────────────────────────────
def read_centroids(path: str) -> np.ndarray:
    """Read centroids produced by reducer.py back into a numpy array."""
    centroids = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cluster_id, coords_str = line.split('\t')
            centroids[int(cluster_id)] = list(map(float, coords_str.split(',')))
    # Return as ordered array (cluster 0, 1, ..., K-1)
    return np.array([centroids[i] for i in sorted(centroids)])


# Write initial centroids for iteration 0
CENTROID_PATH = 'centroids.txt'       # mapper.py looks for this in cwd
write_centroids(initial_centroids, CENTROID_PATH)
print(f"✓ Initial centroids written → {CENTROID_PATH}")

In [ ]:
import subprocess
import shutil

def run_mapreduce_iteration(tsv_path: str, centroid_path: str, output_path: str) -> str:
    """
    Run one MapReduce iteration using mapper.py and reducer.py via
    Unix pipes — identical to Hadoop Streaming's local execution.

    Hadoop Streaming equivalent:
      hadoop jar hadoop-streaming.jar \\
        -files centroids.txt,mapper.py,reducer.py \\
        -mapper  "python3 mapper.py"  \\
        -reducer "python3 reducer.py" \\
        -input  /hdfs/input/data.tsv  \\
        -output /hdfs/output/iter_N

    Returns: path to the file containing new centroids.
    """
    # Hadoop Streaming sorts mapper output by key before the reducer sees it.
    # We replicate this with `sort -k1,1n`.
    cmd = (
        f"cat {tsv_path} "
        f"| python3 mapper.py "
        f"| sort -k1,1n "
        f"| python3 reducer.py"
    )
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"MapReduce failed:\n{result.stderr}")

    with open(output_path, 'w') as f:
        f.write(result.stdout)

    return output_path


# ── Iterative MapReduce loop ──────────────────────────────────────────────────
centroid_history   = [initial_centroids.copy()]
convergence_deltas = []

current_centroids = initial_centroids.copy()

print("Starting MapReduce K-Means iterations...")
print(f"  K={K}  |  max_iter={MAX_ITER}  |  convergence_threshold={CONVERGENCE}")
print("-" * 60)

for iteration in range(1, MAX_ITER + 1):

    # 1) Write current centroids so mapper.py can load them
    write_centroids(current_centroids, CENTROID_PATH)

    # 2) Run one full Map → Sort → Reduce pass
    iter_output = f'{ITER_DIR}/centroids_iter_{iteration:02d}.txt'
    run_mapreduce_iteration(TSV_PATH, CENTROID_PATH, iter_output)

    # 3) Read new centroids from reducer output
    new_centroids = read_centroids(iter_output)

    # 4) Check convergence — max shift across all centroids
    delta = np.max(np.linalg.norm(new_centroids - current_centroids, axis=1))
    convergence_deltas.append(delta)
    centroid_history.append(new_centroids.copy())

    print(f"  Iter {iteration:02d}  |  max centroid shift = {delta:.6f}")

    current_centroids = new_centroids

    if delta < CONVERGENCE:
        print(f"\n✓ Converged after {iteration} iterations  (Δ = {delta:.2e} < {CONVERGENCE})")
        break
else:
    print(f"\n⚠ Reached max iterations ({MAX_ITER}) without full convergence")

# Save final centroids
final_centroid_path = f'{MR_DIR}/final_centroids.txt'
write_centroids(current_centroids, final_centroid_path)
shutil.copy(final_centroid_path, CENTROID_PATH)   # keep cwd copy up to date
print(f"✓ Final centroids saved → {final_centroid_path}")

## Step 5: Label Assignment

Run the **final mapper** once more to get every player's cluster assignment, then map cluster IDs → human-readable labels based on centroid characteristics.

In [ ]:
# ── Run final mapper to get cluster assignments ───────────────────────────────
cmd = f"cat {TSV_PATH} | python3 mapper.py | sort -k1,1n"
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"Final mapper failed:\n{result.stderr}")

# Parse assignments: cluster_id \t f1,f2,...
assignments = []
for line in result.stdout.strip().split('\n'):
    if not line:
        continue
    cluster_id_str, _ = line.split('\t', 1)
    assignments.append(int(cluster_id_str))

print(f"✓ Cluster assignments retrieved for {len(assignments):,} players")

# Attach to scaled_df
result_df = scaled_df.copy()
result_df['cluster_id'] = assignments

# Cluster size distribution
print("\nCluster sizes:")
print(result_df['cluster_id'].value_counts().sort_index())

In [ ]:
# ── Interpret clusters → map to labels ───────────────────────────────────────
#
# Strategy: compute the mean of each centroid's key offensive features
# (goals, assists, appearances) in the ORIGINAL (unscaled) space so the
# ranking is interpretable. Then sort clusters high→low and assign labels.

# Re-attach original (unscaled) stats for interpretation
ORIGINAL_STATS = ['goals_total', 'assists_total', 'nb_on_pitch_total',
                  'minutes_played_total', 'national_matches_total']
available_stats = [c for c in ORIGINAL_STATS if c in feature_matrix.columns]

interp_df = feature_matrix[['player_id'] + available_stats].copy()
interp_df['cluster_id'] = result_df['cluster_id'].values

cluster_means = interp_df.groupby('cluster_id')[available_stats].mean()

# Composite score: sum of z-scored cluster means across key dimensions
from sklearn.preprocessing import MinMaxScaler
mm = MinMaxScaler()
cluster_scores = pd.Series(
    mm.fit_transform(cluster_means).sum(axis=1),
    index=cluster_means.index
)

# Sort clusters by composite score → assign label tier
sorted_clusters = cluster_scores.sort_values(ascending=False).index.tolist()

LABEL_MAP = {
    sorted_clusters[0]: 'Elite',
    sorted_clusters[1]: 'Star',
    sorted_clusters[2]: 'Reliable Player',
    sorted_clusters[3]: 'Squad Player',
    sorted_clusters[4]: 'Low Impact',
}

print("Cluster → Label mapping:")
for cid, label in sorted(LABEL_MAP.items()):
    score = cluster_scores[cid]
    print(f"  Cluster {cid}  →  {label:<18}  (composite score: {score:.3f})")

# Apply labels
result_df['player_label'] = result_df['cluster_id'].map(LABEL_MAP)
print("\n✓ Labels assigned")

In [ ]:
# ── Merge labels back to original player info ─────────────────────────────────
final_df = X_df[['player_id', 'player_slug']].merge(
    result_df[['player_id', 'cluster_id', 'player_label']],
    on='player_id',
    how='inner'
)

# Also attach key raw stats for the output report
report_stats = ['goals_total', 'assists_total', 'nb_on_pitch_total',
                'minutes_played_total', 'national_matches_total', 'national_goals_total']
available_report = [c for c in report_stats if c in X_df.columns]

final_df = final_df.merge(
    X_df[['player_id'] + available_report],
    on='player_id',
    how='left'
)

# Save results
RESULTS_PATH = f'{OUTPUT_DIR}/player_categories.csv'
final_df.to_csv(RESULTS_PATH, index=False)
print(f"✓ Results saved → {RESULTS_PATH}")
print(f"  Shape: {final_df.shape}")

# Label distribution
print("\nLabel distribution:")
label_order = ['Elite', 'Star', 'Reliable Player', 'Squad Player', 'Low Impact']
dist = final_df['player_label'].value_counts().reindex(label_order)
total = dist.sum()
for label, count in dist.items():
    bar = '█' * int(count / total * 40)
    print(f"  {label:<18} {count:>6,}  {bar}")

final_df.head(10)

## Step 6: Evaluation & Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

sns.set_theme(style='whitegrid', font_scale=1.1)
LABEL_COLORS = {
    'Elite':           '#FFD700',   # gold
    'Star':            '#1E90FF',   # dodger blue
    'Reliable Player': '#32CD32',   # lime green
    'Squad Player':    '#FF8C00',   # dark orange
    'Low Impact':      '#A9A9A9',   # grey
}

# ── Clustering quality metrics ────────────────────────────────────────────────
cluster_labels_arr = result_df['cluster_id'].values

sil  = silhouette_score(data_points, cluster_labels_arr, sample_size=min(10_000, len(data_points)), random_state=42)
db   = davies_bouldin_score(data_points, cluster_labels_arr)
ch   = calinski_harabasz_score(data_points, cluster_labels_arr)

print("Clustering Quality Metrics")
print("─" * 40)
print(f"  Silhouette Score        : {sil:.4f}   (higher → better, range −1 to 1)")
print(f"  Davies-Bouldin Index    : {db:.4f}   (lower  → better)")
print(f"  Calinski-Harabasz Index : {ch:.1f}  (higher → better)")

In [ ]:
# ── Plot 1: Convergence curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(convergence_deltas) + 1), convergence_deltas,
        marker='o', color='steelblue', linewidth=2)
ax.axhline(CONVERGENCE, color='red', linestyle='--', linewidth=1, label=f'Threshold ({CONVERGENCE})')
ax.set_xlabel('MapReduce Iteration')
ax.set_ylabel('Max Centroid Shift')
ax.set_title('K-Means Convergence (MapReduce)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/convergence_curve.png', dpi=150)
plt.show()
print("✓ Convergence curve saved")

In [ ]:
# ── Plot 2: Label distribution bar chart ─────────────────────────────────────
label_order = ['Elite', 'Star', 'Reliable Player', 'Squad Player', 'Low Impact']
counts = final_df['player_label'].value_counts().reindex(label_order)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(counts.index, counts.values,
              color=[LABEL_COLORS[l] for l in counts.index],
              edgecolor='black', linewidth=0.7)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Player Category Distribution (K-Means MapReduce, K=5)', fontsize=13)
ax.set_xlabel('Player Label')
ax.set_ylabel('Number of Players')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/label_distribution.png', dpi=150)
plt.show()
print("✓ Label distribution chart saved")

In [ ]:
# ── Plot 3: PCA 2-D cluster scatter ──────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(data_points)

plot_df = pd.DataFrame({
    'PC1':   pca_coords[:, 0],
    'PC2':   pca_coords[:, 1],
    'label': result_df['player_label'].values
})

# Sample for readability if dataset is huge
SAMPLE = min(5_000, len(plot_df))
plot_sample = plot_df.sample(SAMPLE, random_state=42)

fig, ax = plt.subplots(figsize=(10, 7))
for label in label_order:
    mask = plot_sample['label'] == label
    ax.scatter(plot_sample.loc[mask, 'PC1'],
               plot_sample.loc[mask, 'PC2'],
               c=LABEL_COLORS[label], label=label,
               alpha=0.55, s=18, edgecolors='none')

# Plot centroids projected onto PCA space
centroid_pca = pca.transform(current_centroids)
for cid, (cx, cy) in enumerate(centroid_pca):
    lbl = LABEL_MAP[cid]
    ax.scatter(cx, cy, marker='*', s=300, c=LABEL_COLORS[lbl],
               edgecolors='black', linewidth=0.8, zorder=5)

ax.set_title(f'Player Clusters — PCA Projection (K=5, n={SAMPLE:,} sampled)', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(title='Player Label', loc='best')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/pca_cluster_scatter.png', dpi=150)
plt.show()
print("✓ PCA scatter plot saved")

In [ ]:
# ── Plot 4: Centroid radar / heatmap ─────────────────────────────────────────
# Show the average feature profile of each label tier

centroid_df = pd.DataFrame(current_centroids, columns=CLUSTERING_FEATURES)
centroid_df.index = [LABEL_MAP[i] for i in range(K)]
centroid_df = centroid_df.loc[label_order]   # consistent order

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    centroid_df,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Cluster Centroids — Feature Heatmap (Standardised Scale)', fontsize=13)
ax.set_xlabel('Feature')
ax.set_ylabel('Player Label')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/centroid_heatmap.png', dpi=150)
plt.show()
print("✓ Centroid heatmap saved")

In [ ]:
# ── Plot 5: Box plots — Goals & Appearances per label ────────────────────────
plot_stats = [c for c in ['goals_total', 'assists_total', 'nb_on_pitch_total'] if c in final_df.columns]

if plot_stats:
    fig, axes = plt.subplots(1, len(plot_stats), figsize=(5 * len(plot_stats), 5))
    if len(plot_stats) == 1:
        axes = [axes]

    for ax, stat in zip(axes, plot_stats):
        ordered_data = [final_df.loc[final_df['player_label'] == lbl, stat].dropna() for lbl in label_order]
        bp = ax.boxplot(ordered_data, patch_artist=True, notch=False,
                        medianprops={'color': 'black', 'linewidth': 2})
        for patch, lbl in zip(bp['boxes'], label_order):
            patch.set_facecolor(LABEL_COLORS[lbl])
            patch.set_alpha(0.8)
        ax.set_xticklabels(label_order, rotation=20, ha='right')
        ax.set_title(stat.replace('_total', '').replace('_', ' ').title())
        ax.set_ylabel('Value')

    fig.suptitle('Feature Distribution by Player Label', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/boxplots_by_label.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Box plots saved")

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 60)
print("     PLAYER CATEGORISATION — FINAL SUMMARY")
print("=" * 60)
print(f"  Algorithm     : K-Means via MapReduce (Hadoop Streaming)")
print(f"  K (clusters)  : {K}")
print(f"  Iterations    : {len(convergence_deltas)}")
print(f"  Total players : {len(final_df):,}")
print()
print("  Label breakdown:")
for lbl in label_order:
    n = (final_df['player_label'] == lbl).sum()
    pct = n / len(final_df) * 100
    print(f"    {lbl:<18} {n:>6,}  ({pct:.1f}%)")
print()
print(f"  Silhouette Score     : {sil:.4f}")
print(f"  Davies-Bouldin Index : {db:.4f}")
print(f"  Calinski-Harabasz   : {ch:.1f}")
print()
print(f"  Output files saved to: {OUTPUT_DIR}/")
print("=" * 60)